In [ ]:
from typing import TypedDict, Literal, Sequence
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from rich import print

load_dotenv(override=True)

CONTENT_TYPES = ["poem", "joke", "ci_poem"]

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を定義
#1.1 グローバル状態
class OverAllState(TypedDict):
    topic: str
    poem: str
    ci_poem: str
    joke: str


#1.2 プライベート状態
class workerState(TypedDict):
    content_type: Literal["poem", "joke", "ci_poem"]
    prompt: str


#1.3 入力状態
class inputState(TypedDict):
    topic: str


#1.4 出力状態
class outputState(TypedDict):
    poem: str
    ci_poem: str
    joke: str


# 2. ノードを定義
def worker_node(state: workerState) -> outputState:
    content_type = state["content_type"]
    prompt = state["prompt"]
    content = model.invoke([HumanMessage(prompt)]).content
    return {
        content_type: content
    }


#3. 動的分岐のルーターを定義
def router(state: inputState) -> Sequence[Send]:
    router_prompt = "{} をテーマにした{}を生成してください"
    english2Chinese = {
        "poem": "俳句",
        "joke": "ジョーク",
        "ci_poem": "短歌"
    }
    topic = state["topic"]
    return [
        Send(
            "worker_node",
            {
                "content_type": content_type,
                "prompt": router_prompt.format(topic, english2Chinese[content_type])
            }
        )
        for content_type in CONTENT_TYPES
    ]


#4. グラフを構築
builder = StateGraph(state_schema=OverAllState, input_schema=inputState, output_schema=outputState)

builder.add_node("worker_node", worker_node)
builder.add_conditional_edges(
    START,
    router,
    path_map=["worker_node"],
)

builder.add_edge("worker_node", END)

graph = builder.compile()
res = graph.invoke({"topic": "桜"})
print(res)

from IPython.display import display

display(graph)


